# 01 - Data Loading & Initial Exploration

> **Objective:** Load the Bank Marketing dataset and perform initial inspection to understand structure, quality, and characteristics before any transformation.

---

## 1.1 Import Libraries

We begin by importing the core libraries for data manipulation and numerical computation.

In [1]:
import pandas as pd
import numpy as np




**Why these libraries?**
- **Pandas** — Industry-standard for tabular data manipulation (loading, filtering, grouping, transformation)
- **NumPy** — Efficient numerical operations and array handling; backend for Pandas and Scikit-learn

---

## 1.2 Load the Dataset

The raw dataset is preserved in `data/raw/` to maintain an immutable source of truth. We load it using a relative path for portability across environments.

In [2]:
df = pd.read_csv("../data/raw/bank.csv")


**Key decision:** Using `../data/raw/bank.csv` ensures the notebook works on any machine that clones the repository, without hardcoding absolute paths.

---

## 1.3 First Look: Sample Records

Before any analysis, we inspect the first few rows to validate that the data loaded correctly and to form an initial mental model of the features.

In [3]:
df.head()


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


**What to observe:**
- Column names and their semantic meaning
- Sample values (categorical vs. numerical)
- Presence of placeholder values like `"unknown"`
- The target variable (`deposit`: `yes` / `no`)

---

## 1.4 Extended Preview

A slightly larger sample helps detect early inconsistencies that might not appear in the first 5 rows.

In [4]:
df.head(10)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes
5,42,management,single,tertiary,no,0,yes,yes,unknown,5,may,562,2,-1,0,unknown,yes
6,56,management,married,tertiary,no,830,yes,yes,unknown,6,may,1201,1,-1,0,unknown,yes
7,60,retired,divorced,secondary,no,545,yes,no,unknown,6,may,1030,1,-1,0,unknown,yes
8,37,technician,married,secondary,no,1,yes,no,unknown,6,may,608,1,-1,0,unknown,yes
9,28,services,single,secondary,no,5090,yes,no,unknown,6,may,1297,3,-1,0,unknown,yes


## 1.5 Dataset Dimensions

Understanding the scale of the data informs downstream decisions on model complexity, cross-validation strategy, and computational requirements.

In [5]:
df.shape


(11162, 17)

**Interpretation:**
- **Rows** = Number of customer records (observations)
- **Columns** = Number of features + target variable
  
---

## 1.6 Structure & Data Types

`df.info()` reveals the formal structure: data types, non-null counts, and memory footprint. This is critical for detecting hidden missing values (e.g., `"unknown"` strings that are not `NaN`) and type mismatches.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        11162 non-null  int64 
 1   job        11162 non-null  object
 2   marital    11162 non-null  object
 3   education  11162 non-null  object
 4   default    11162 non-null  object
 5   balance    11162 non-null  int64 
 6   housing    11162 non-null  object
 7   loan       11162 non-null  object
 8   contact    11162 non-null  object
 9   day        11162 non-null  int64 
 10  month      11162 non-null  object
 11  duration   11162 non-null  int64 
 12  campaign   11162 non-null  int64 
 13  pdays      11162 non-null  int64 
 14  previous   11162 non-null  int64 
 15  poutcome   11162 non-null  object
 16  deposit    11162 non-null  object
dtypes: int64(7), object(10)
memory usage: 1.4+ MB



**What we verify:**
- No unexpected `NaN` values (all non-null counts equal to row count)
- Correct data type assignments (`object` for categorical, `int64` for numerical)
- Memory efficiency (relevant if scaling to larger datasets)

---

## 1.7 Statistical Summary

Descriptive statistics for numerical features reveal distribution shape, spread, and potential outliers before visualization.


In [7]:
df.describe()


,age,balance,day,duration,campaign,pdays,previous
count,11162.000000,11162.000000,11162.000000,11162.000000,11162.000000,11162.000000,11162.000000
mean,41.231948,1528.538524,15.658036,371.993818,2.508421,51.330407,0.832557
std,11.913369,3225.413326,8.420740,347.128386,2.722077,108.758282,2.292007
min,18.000000,-6847.000000,1.000000,2.000000,1.000000,-1.000000,0.000000
25%,32.000000,122.000000,8.000000,138.000000,1.000000,-1.000000,0.000000
50%,39.000000,550.000000,15.000000,255.000000,2.000000,-1.000000,0.000000
75%,49.000000,1708.000000,22.000000,496.000000,3.000000,20.750000,1.000000
max,95.000000,81204.000000,31.000000,3881.000000,63.000000,854.000000,58.000000


**Signals to watch for:**
- **Mean vs. Median (50%)** divergence → indicates skewness (common in `balance`)
- **Large min-max gaps** → potential outliers requiring IQR or capping treatment
- **High standard deviation** → high variability in features like `campaign` or `balance`

---

## Summary & Next Steps

| Checkpoint | Status | Key Finding |
|------------|--------|-------------|
| Data loaded | ✅ | `bank.csv` successfully ingested |
| No null values | ✅ | All columns report full non-null counts |
| Target variable | ✅ | `deposit` (binary: `yes`/`no`) confirmed |
| Data types | ✅ | Mix of categorical (`object`) and numerical (`int64`) |
| Outlier risk | ⚠️ | `balance` and `campaign` show wide ranges; require deeper inspection |

**Next:** Proceed to `02_data_cleaning.ipynb` to handle outliers, encode categorical variables, and prepare the dataset for modeling.

---